# Project 2:  Markov Chain Text Generation

In [342]:
import string
import numpy as np
from pprint import pprint

## build_markov_model (1st order, generalized to Nth order)

In [343]:
def build_markov_model(markov_model, text , order=1):
    """
    Args:
        markov_model (dict of dicts): existing model to add to
        text (str): text to learn from
        order (int): how many previous words to use as the state

    Returns:
        markov_model (dict of dicts): updated model
    """

    if markov_model == None: markov_model = dict()

    # Splits current sonnet into separate words. Also removing punctuation from text
    clean_text = text.translate(str.maketrans('', '', string.punctuation))
    sonnet_words = clean_text.split()

    # Add start and end states to the sonnet words.
    for i in range(0, order):
        sonnet_words.insert(0, '*S*')
    sonnet_words.append('*E*')

    # Loops through the current sonnet and creates a Markov model based on the order specified.
    for i in range(len(sonnet_words)):
        if i + order < len(sonnet_words):
            next_word = sonnet_words[i + order]
        else:
            break

        if tuple(sonnet_words[i:i + order]) not in markov_model.keys():
            markov_model[tuple(sonnet_words[i:i + order])] = dict()

        if next_word not in markov_model[tuple(sonnet_words[i:i + order])].keys():
            markov_model[tuple(sonnet_words[i:i + order])][next_word] = 0

        markov_model[tuple(sonnet_words[i:i + order])][next_word] += 1

    # Returns a dictionary of dictionaries containing the states as keys and dictionaries of words and their frequencies as key-value pairs as values.
    return markov_model


In [344]:
# Test: check build_markov_model against the "one fish two fish" example
test_model = build_markov_model({}, "one fish two fish red fish blue fish", order=1)
pprint(test_model)
#Confirmed:  returns {'*S*': {'one': 1}, 'one': {'fish': 1}, 'fish': {'two': 1, 'red': 1, 'blue': 1, '*E*': 1}, 'two': {'fish': 1}, 'red': {'fish': 1}, 'blue': {'fish': 1}}

{('*S*',): {'one': 1},
 ('blue',): {'fish': 1},
 ('fish',): {'*E*': 1, 'blue': 1, 'red': 1, 'two': 1},
 ('one',): {'fish': 1},
 ('red',): {'fish': 1},
 ('two',): {'fish': 1}}


## get_next_word

In [345]:
def get_next_word(current_state, markov_model, seed=42):
    """
    Args:
        current_state: the current word (or tuple of words) to look up
        markov_model (dict of dicts): the model
        seed (int): random seed for reproducibility

    Returns:
        next_word (str): randomly selected next word
    """
    #Look up the current (outer) word as a key in the dictionary, to get the dictionary of possible next (inner) words and each next word is paired with a count of how many times it followed the current word.
    next_word_counts = markov_model[current_state]
    #Add up all the counts for this current word to get a total.
    total = sum(next_word_counts.values())

    #Create empty dictionary to hold the possible next words and their matching probabilities as key-value pairs.
    next_word_probabilities = {}

    #For each possible next word, divide its count by the total to get a probability.
    for word in next_word_counts:
        probability = next_word_counts[word] / total  #Divide this word's count by the total to get its probability
        next_word_probabilities[word] = probability  #Add this word and probability to our list

    #Using those probabilities as weights, randomly select one next word (using the given seed so results are reproducible).  Return the selected next word.
    np.random.seed(seed)  #Set the random seed so results are reproducible
    next_word = np.random.choice(list(next_word_probabilities.keys()), p=list(next_word_probabilities.values())) #Randomly select a word

    return next_word  #Return the selected word.



In [346]:
#Test: build a tiny dictionary to check that get_next_word function works
test_model = {"fish": {"two": 1, "red": 1, "blue": 1, "*E*": 1}}
print(get_next_word("fish", test_model, seed=42))
#Confirmed:  returns "red" with seed=42

red


## generate_random_text

In [347]:
def generate_random_text(markov_model, seed=42):
    """
    Args:
        markov_model (dict of dicts): the model
        seed (int): random seed for reproducibility

    Returns:
        sentence (str): generated text
    """
    #Initialize the initial state and generated sentence strings
    current_word = next(iter(markov_model))
    generated_sentence = ''
	#Randomly selects a word from the starting state probabilities in the Markov chain model and daisy chains words by randomly selecting them based on the previous word until an end state word is selected.
    while '*E*' not in current_word and len(generated_sentence) < 1000:
        next_word = get_next_word(current_word, markov_model, seed=seed)
        if next_word == '*E*':
            break
        generated_sentence = generated_sentence + next_word + ' '
        current_word = current_word[1:] + (next_word,)
	#Returns the generated sentence.
    return generated_sentence
	#Creates a safety cap in case the model never selects an end state. Since words are randomly generated, it is possible that it will keep looping through the Markov model without reaching an end state. The maximum word count prevents the program from continuing indefinitely.


## Pick Your Poison: Sonnets

In [348]:
poison_markov_model = dict()
with open("data/sonnets.txt", "r") as poison_text:
    # Process the lines. Consider that sonnets are separated by an empty line. and create corpus.
    corpus = []
    sonnet = ''
    for line in poison_text:
        if line != '\n':
            sonnet = sonnet + line
        else:
            corpus.append(sonnet)
            sonnet = ''
    if sonnet:
        corpus.append(sonnet)
    for sonnet in corpus:
        poison_markov_model = build_markov_model(markov_model = poison_markov_model, text = sonnet, order=2)
print(generate_random_text(poison_markov_model, seed=42))

So are you made That millions of strange shadows on you tend Since every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath every one hath eve